# US Baby Names Dashboard


In [68]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sqlite3

conn = sqlite3.connect("../data/cleaned/names_clean.sqlite")
# Hinweis: df_national/df_state werden hier bewusst NICHT vollstaendig geladen -
# alle Charts fragen direkt und aggregiert per SQL ueber "conn" ab (schneller, weniger RAM).


In [69]:
# Farbpalette
DARK = "#5D3140"
PINK = "#CF4173"
LIGHT_PINK = "#F39399"
BEIGE = "#F6D8BD"


In [70]:
query_top10 = """
SELECT Name, SUM(Count) AS Total
FROM NationalNames
WHERE Gender = 'F'
GROUP BY Name
ORDER BY Total DESC
LIMIT 10;
"""

top10_female = pd.read_sql_query(query_top10, conn)


In [71]:
query_top10_male = """
SELECT Name, SUM(Count) AS Total
FROM NationalNames
WHERE Gender = 'M'
GROUP BY Name
ORDER BY Total DESC
LIMIT 10;
"""

top10_male = pd.read_sql_query(query_top10_male, conn)


In [72]:
query_mary = """
SELECT Year, SUM(Count) AS Births
FROM NationalNames
WHERE Name = 'Mary'
GROUP BY Year
ORDER BY Year;
"""

mary = pd.read_sql_query(query_mary, conn)


In [73]:
query_seth = """
SELECT
    s.State,
    SUM(s.Count) AS SethBirths,
    t.TotalBirths,
    10000.0 * SUM(s.Count) / t.TotalBirths AS SethPer10000
FROM StateNames s
JOIN (
    SELECT State, SUM(Count) AS TotalBirths
    FROM StateNames
    GROUP BY State
) t
ON s.State = t.State
WHERE s.Name = 'Seth'
GROUP BY s.State
ORDER BY SethPer10000 DESC;
"""

seth = pd.read_sql_query(query_seth, conn)


In [74]:
query_first_last = """
SELECT *
FROM (
    SELECT MIN(Year) AS Year, Name, SUM(Count) AS Births
    FROM NationalNames
    WHERE Year = (SELECT MIN(Year) FROM NationalNames)
    GROUP BY Name
    ORDER BY Births DESC
    LIMIT 3
);
UNION ALL
SELECT *
FROM (
    SELECT MAX(Year) AS Year, Name, SUM(Count) AS Births
    FROM NationalNames
    WHERE Year = (SELECT MAX(Year) FROM NationalNames)
    GROUP BY Name
    ORDER BY Births DESC
    LIMIT 3
);
"""

top_first_last = pd.read_sql_query(query_first_last, conn)
top_first_last


DatabaseError: Execution failed on sql '
SELECT *
FROM (
    SELECT MIN(Year) AS Year, Name, SUM(Count) AS Births
    FROM NationalNames
    WHERE Year = (SELECT MIN(Year) FROM NationalNames)
    GROUP BY Name
    ORDER BY Births DESC
    LIMIT 3
);
UNION ALL
SELECT *
FROM (
    SELECT MAX(Year) AS Year, Name, SUM(Count) AS Births
    FROM NationalNames
    WHERE Year = (SELECT MAX(Year) FROM NationalNames)
    GROUP BY Name
    ORDER BY Births DESC
    LIMIT 3
);
': You can only execute one statement at a time.

In [ ]:
fig = make_subplots(
    rows=5,
    cols=2,
    specs=[
        [{"type": "bar", "colspan": 2}, None],
        [{"type": "bar", "colspan": 2}, None],
        [{"type": "scatter", "colspan": 2}, None],
        [{"type": "choropleth", "colspan": 2}, None],
        [{"type": "bar"}, {"type": "bar"}]
    ],
    row_heights=[0.18, 0.18, 0.20, 0.30, 0.19],
    vertical_spacing=0.07,
    subplot_titles=(
        "Top 10 Female Names",
        "Top 10 Male Names",
        "Popularity of the Name 'Mary' Over Time",
        "Popularity of 'Seth' Across the United States",
        "First Recorded Year",
        "Last Recorded Year"
    )
);


In [ ]:
fig.add_trace(
    go.Bar(
        x=top10_female["Total"],
        y=top10_female["Name"],
        orientation="h",
        marker=dict(color=PINK),
        text=top10_female["Total"],
        texttemplate="%{text:.3s}",
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Births: %{x:,.0f}<extra></extra>"
    ),
    row=1, col=1
);

fig.update_yaxes(autorange="reversed", row=1, col=1);


In [ ]:
fig.add_trace(
    go.Bar(
        x=top10_male["Total"],
        y=top10_male["Name"],
        orientation="h",
        marker=dict(color=DARK),
        text=top10_male["Total"],
        texttemplate="%{text:.3s}",
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Births: %{x:,.0f}<extra></extra>"
    ),
    row=2, col=1
);

fig.update_yaxes(autorange="reversed", row=2, col=1);


In [ ]:
fig.add_trace(
    go.Scatter(
        x=mary["Year"],
        y=mary["Births"],
        mode="lines",
        name="Mary",
        line=dict(color=DARK, width=3),
        fill="tozeroy",
        fillcolor="rgba(243, 147, 153, 0.25)",
        hovertemplate="<b>Year:</b> %{x}<br><b>Births:</b> %{y:,.0f}<extra></extra>"
    ),
    row=3, col=1
);


In [ ]:
fig.add_trace(
    go.Choropleth(
        locations=seth["State"],
        z=seth["SethPer10000"],
        locationmode="USA-states",
        colorscale=[
            [0.00, BEIGE],
            [0.35, LIGHT_PINK],
            [0.70, PINK],
            [1.00, DARK]
        ],
        marker_line_color="white",
        marker_line_width=0.7,
        colorbar=dict(title=dict(text="Seth per<br>10,000 births")),
        hovertemplate="<b>%{location}</b><br>Seth per 10,000 births: %{z:.1f}<extra></extra>"
    ),
    row=4, col=1
);

fig.update_geos(scope="usa", projection_type="albers usa", row=4, col=1);


In [ ]:
first_year = top_first_last[
    top_first_last["Year"] == top_first_last["Year"].min()
].sort_values("Births")

fig.add_trace(
    go.Bar(
        x=first_year["Births"],
        y=first_year["Name"],
        orientation="h",
        marker=dict(color=LIGHT_PINK),
        text=first_year["Births"],
        texttemplate="%{text:.3s}",
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Births: %{x:,.0f}<extra></extra>",
        name=str(first_year["Year"].iloc[0])
    ),
    row=5, col=1
);

fig.update_yaxes(autorange="reversed", row=5, col=1);


In [ ]:
last_year = top_first_last[
    top_first_last["Year"] == top_first_last["Year"].max()
].sort_values("Births")

fig.add_trace(
    go.Bar(
        x=last_year["Births"],
        y=last_year["Name"],
        orientation="h",
        marker=dict(color=PINK),
        text=last_year["Births"],
        texttemplate="%{text:.3s}",
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Births: %{x:,.0f}<extra></extra>",
        name=str(last_year["Year"].iloc[0])
    ),
    row=5, col=2
);


In [ ]:
fig.update_layout(
    title={"text": "US Baby Names Dashboard", "x": 0.5, "xanchor": "center"},
    height=1750,
    showlegend=False,
    margin=dict(l=60, r=60, t=100, b=50),
    template="plotly_white"
);
# Hinweis: der urspruengliche "colorscale"-Block hier wurde entfernt -
# "colorscale" ist kein gueltiges Layout-Property und verursachte einen ValueError.
# Die Farbskala fuer die Karte ist bereits am Choropleth-Trace selbst gesetzt (siehe oben).


In [ ]:
fig.update_xaxes(title_text="Total Births", tickformat=",", row=1, col=1);
fig.update_xaxes(title_text="Total Births", tickformat=",", row=2, col=1);
fig.update_xaxes(title_text="Year", row=3, col=1);
fig.update_yaxes(title_text="Births", tickformat=",", row=3, col=1);
fig.update_xaxes(title_text="Births", tickformat=",", row=5, col=1);
fig.update_xaxes(title_text="Births", tickformat=",", row=5, col=2);


In [75]:
fig.show()


In [ ]:
conn.close()
